# Fix hidden-slide page numbers in Unstructured PPTX output

When a PPTX goes through Unstructured's **VLM partitioner**, hidden slides are dropped and the remaining slides are renumbered 1..N. Page numbers in the output then no longer match the deck.

This notebook works on **any number of decks and their Partitioner / Chunker JSONs**:
1. Finds every JSON in `JSON_FOLDER` and works out which deck it came from (from `metadata.filename`, which Unstructured writes into every element)
2. Finds that deck in `PPTX_FOLDER` and reads which slides are hidden
3. Rewrites `page_number`, `data-page-number` in `text_as_html`, and the same fields inside `orig_elements`
4. Saves corrected copies as `<name>-fixed.json` and checks the result

Original files are never modified. The notebook, the decks and the JSONs can all be in different folders.

**Only the cell below needs editing.**

## 1. Settings

In [ ]:
from pathlib import Path

# Folder containing the Partitioner / Chunker JSONs (subfolders are searched too)
JSON_FOLDER = Path(r"")

# Folder containing the original .pptx decks (subfolders are searched too).
# Can be the same as JSON_FOLDER.
PPTX_FOLDER = Path(r"")

# Where to write the *-fixed.json files. None = next to each original JSON.
OUTPUT_FOLDER = None

## 2. Helper functions

In [ ]:
import base64, gzip, json, re

try:
    from pptx import Presentation
except ImportError:
    raise ImportError("python-pptx is missing. Run:  %pip install python-pptx  then restart the kernel.")

PAGE_RE = re.compile(r'(data-page-number=\\?")(\d+)(\\?")')
CHUNK_TYPES = {"CompositeElement", "TableChunk"}

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_orig(md):
    """Return (orig_elements list, was_compressed). Handles base64-gzipped strings."""
    orig = md.get("orig_elements") or []
    if isinstance(orig, str):
        return json.loads(gzip.decompress(base64.b64decode(orig))), True
    return orig, False

def slide_map(pptx_path):
    """Return (total_slides, hidden_slide_numbers, {processed_page: real_slide})."""
    prs = Presentation(pptx_path)
    hidden, mapping, visible_idx = [], {}, 0
    for n, slide in enumerate(prs.slides, start=1):
        if slide._element.get("show") in ("0", "false"):
            hidden.append(n)
        else:
            visible_idx += 1
            mapping[visible_idx] = n
    return len(prs.slides), hidden, mapping

def fix_element(el, mapping, missing):
    md = el.get("metadata") or {}
    pn = md.get("page_number")
    if pn is not None:
        if pn in mapping:
            md["page_number"] = mapping[pn]
        else:
            missing.add(pn)
    html = md.get("text_as_html")
    if isinstance(html, str):
        md["text_as_html"] = PAGE_RE.sub(
            lambda m: f"{m.group(1)}{mapping.get(int(m.group(2)), int(m.group(2)))}{m.group(3)}", html)
    if md.get("orig_elements"):
        items, compressed = read_orig(md)
        for sub in items:
            fix_element(sub, mapping, missing)
        md["orig_elements"] = (base64.b64encode(gzip.compress(json.dumps(items).encode())).decode()
                               if compressed else items)

def pages_in_chunk(ch):
    md = ch.get("metadata") or {}
    pages = {int(m[1]) for m in PAGE_RE.findall(md.get("text_as_html") or "")}
    items, _ = read_orig(md)
    pages |= {(e.get("metadata") or {}).get("page_number") for e in items} - {None}
    return sorted(pages)

def describe_json(path):
    """Return (source_filename, kind) or (None, reason) if not an Unstructured PPTX output."""
    try:
        data = load(path)
    except Exception as e:
        return None, f"unreadable JSON ({e.__class__.__name__})"
    if not isinstance(data, list) or not data or not isinstance(data[0], dict):
        return None, "not an Unstructured element list"
    names = {(e.get("metadata") or {}).get("filename") for e in data} - {None}
    if not names:
        return None, "no metadata.filename"
    name = sorted(names)[0]
    if not name.lower().endswith((".pptx", ".pptm", ".potx")):
        return None, f"source is not a PowerPoint file ({name})"
    kind = "Chunker" if any(e.get("type") in CHUNK_TYPES for e in data) else "Partitioner"
    return name, kind

## 3. Find JSONs and match them to decks

In [ ]:
for label, folder in [("JSON_FOLDER", JSON_FOLDER), ("PPTX_FOLDER", PPTX_FOLDER)]:
    if not str(folder).strip() or str(folder) == "." or not folder.is_dir():
        raise ValueError(f"{label} is not set or does not exist: '{folder}'. Edit the Settings cell.")

decks = {}
for p in PPTX_FOLDER.rglob("*"):
    if p.suffix.lower() in (".pptx", ".pptm", ".potx") and not p.name.startswith("~$"):
        decks.setdefault(p.name, p)

jobs, skipped = [], []
for jp in sorted(JSON_FOLDER.rglob("*.json")):
    if jp.stem.endswith("-fixed"):
        continue
    src, kind = describe_json(jp)
    if src is None:
        skipped.append((jp.name, kind)); continue
    deck = decks.get(src)
    if deck is None:
        # Unstructured sometimes prefixes the stored name; fall back to a suffix match
        deck = next((d for n, d in decks.items() if src.endswith(n) or n.endswith(src)), None)
    if deck is None:
        skipped.append((jp.name, f"deck '{src}' not found in PPTX_FOLDER")); continue
    jobs.append({"json": jp, "kind": kind, "deck": deck})

print(f"Decks found: {len(decks)} | JSONs matched: {len(jobs)} | skipped: {len(skipped)}\n")
for j in jobs:
    print(f"  [{j['kind']:<11}] {j['json'].name}\n                -> {j['deck'].name}")
if skipped:
    print("\nSkipped:")
    for name, why in skipped:
        print(f"  {name}: {why}")

## 4. Hidden slides per deck

Check these against what you see in PowerPoint (hidden slides show a crossed-out number).

In [ ]:
deck_info = {}
for deck in sorted({j["deck"] for j in jobs}):
    total, hidden, mapping = slide_map(deck)
    deck_info[deck] = (total, hidden, mapping)
    print(f"{deck.name}\n  {total} slides | hidden: {hidden or 'none'} | visible: {len(mapping)}")
    shifted = [f"{p}->{s}" for p, s in mapping.items() if p != s]
    if shifted:
        print("  processed page -> real slide: " + ", ".join(shifted))
    print()

## 5. Fix the JSONs

A file is **skipped** if its highest page number is greater than the deck's visible slide count: that means hidden slides were *not* dropped for it (e.g. it went through High Res), and remapping would make it wrong.

In [ ]:
results = []
for j in jobs:
    total, hidden, mapping = deck_info[j["deck"]]
    elements = load(j["json"])
    pages = {(e.get("metadata") or {}).get("page_number") for e in elements} - {None}
    top = max(pages) if pages else 0
    status, out = "", None
    if not hidden:
        status = "no hidden slides - nothing to fix"
    elif top > len(mapping):
        status = f"SKIPPED - highest page {top} > {len(mapping)} visible slides; check manually"
    else:
        missing = set()
        for el in elements:
            fix_element(el, mapping, missing)
        folder = Path(OUTPUT_FOLDER) if OUTPUT_FOLDER else j["json"].parent
        out = folder / f"{j['json'].stem}-fixed.json"
        save(elements, out)
        status = f"fixed -> {out}"
        if missing:
            status += f"  (WARNING: pages with no matching slide: {sorted(missing)})"
    results.append({**j, "fixed": out, "hidden": hidden})
    print(f"[{j['kind']:<11}] {j['json'].name}\n              {status}\n")

## 6. Verify

- Partitioner: no page number should be a hidden slide.
- Chunker: no chunk should contain more than one slide. This is only guaranteed if the Chunker node used **Chunk by page**; with `orig_elements` turned on the check is exact.

In [ ]:
for r in results:
    if not r["fixed"]:
        continue
    data = load(r["fixed"])
    print(f"{r['fixed'].name}")
    if r["kind"] == "Partitioner":
        used = sorted({(e.get("metadata") or {}).get("page_number") for e in data} - {None})
        clash = sorted(set(used) & set(r["hidden"]))
        print(f"  pages {used[0]}..{used[-1]} ({len(used)} distinct) | hidden slides used: {clash or 'none - good'}")
    else:
        bad = [(i, ch["metadata"].get("page_number"), pages_in_chunk(ch))
               for i, ch in enumerate(data, 1) if len(pages_in_chunk(ch)) > 1]
        has_orig = any((ch.get("metadata") or {}).get("orig_elements") for ch in data)
        print(f"  {len(data)} chunks | orig_elements: {has_orig} | spanning >1 slide: {len(bad)}")
        for i, labelled, pages in bad[:20]:
            print(f"    chunk #{i}: labelled {labelled}, contains slides {pages}")
    print()

## 7. Spot-check (optional)

Set `DECK_NAME` (any part of the file name) and `SLIDE`, then compare the printed text with that slide in PowerPoint.

In [ ]:
DECK_NAME = ""   # e.g. part of the deck file name; leave "" to use the first fixed Chunker file
SLIDE = 1

target = next((r for r in results if r["fixed"] and r["kind"] == "Chunker"
               and DECK_NAME.lower() in r["deck"].name.lower()), None)
if target is None:
    print("No fixed Chunker file matches DECK_NAME.")
else:
    print(f"{target['deck'].name} - slide {SLIDE}\n")
    hits = [ch for ch in load(target["fixed"]) if ch["metadata"].get("page_number") == SLIDE]
    for ch in hits:
        print((ch.get("text") or "")[:500], "\n")
    if not hits:
        print("No chunk for this slide (it may be hidden, or have no extractable content).")